In [1]:
import numpy as np
import scipy.signal
import tensorstore as ts
from sklearn.neighbors import KDTree
import matplotlib.pyplot as plt
import asyncio
# from tqdm.asyncio import tqdm_asyncio
from mpl_toolkits.axes_grid1 import ImageGrid

## compare cell activity with zapbench trace

In [2]:
ds = ts.open({
    'open': True,
    'driver': 'zarr3',
    'kvstore': 'file:///groups/saalfeld/home/kumarv4/repos/zapbench/output.zarr/cell_activity_normalized'
}).result()
ds.shape

In [3]:
act_new = ds.read().result()

In [4]:
# Pull data from google storage since the raw data takes a while to download
gs_uri = "gs://zapbench-release/volumes/20240930"
# gs_uri = "file:///groups/saalfeld/saalfeldlab/zapbench-release/volumes/20240930"
ds = ts.open({
    'open': True,
    'driver': 'zarr3',
    'kvstore': f'{gs_uri}/traces'
}).result()
ds.shape

In [5]:
act_zap = ds.read().result()

In [6]:
from zapbench.constants import CONDITION_OFFSETS

In [148]:
cell_id = np.random.randint(71721)
# cell_id=52779
# cell_id=39487
print(f"{cell_id=}")

plt.figure(figsize=(20, 5))
plt.plot(act_zap[:, cell_id],
         label="zapbench"
         )
plt.plot(act_new[cell_id, :], #/ act_new[cell_id, :].mean()
         label="new"
         )
for i in CONDITION_OFFSETS:
  plt.axvline(i, color="k", ls="dotted")
plt.legend()
plt.title(f"{cell_id=}")
plt.xlim(0, 7879)
plt.xlabel("time")
plt.ylabel("dF/F activity")

In [89]:
# cell_id = np.random.randint(71721)
print(f"{cell_id=}")
# plt.scatter(
#     act_zap[:, cell_id] / act_zap[:, cell_id].max(),
#     act_new[cell_id, :] / act_new[cell_id, :].max()
# )
tmin, tmax = CONDITION_OFFSETS[1], CONDITION_OFFSETS[1]+400

plt.figure(figsize=(20, 5))
plt.plot(np.arange(tmin, tmax), act_zap[tmin:tmax, cell_id] )
plt.plot(np.arange(tmin, tmax), act_new[cell_id, tmin:tmax] )
# for i in CONDITION_OFFSETS:
#   plt.axvline(i, color="k", ls="dotted")

plt.xlim(tmin, tmax)

## check whether transform to raw zstack works

In [32]:
# Load segmentation
ds = ts.open({
    'open': True,
    'driver': 'zarr3',
    'kvstore': f'{gs_uri}/segmentation'
}).result()
segmentation = ds.read().result()

In [33]:
# process segmentation to build a map between voxels and cell id's

xi, yi, zi = np.where(segmentation > 0)


# get to zero based indexing
cell_id_flat = segmentation[xi, yi, zi].astype(np.uint64)
cell_id_flat -= 1


In [34]:

# ============================================================================
# FLOW FIELDS SCHEMA
# ============================================================================
# Shape: [3, 36, 83, 128, 7879] with dimensions ["fc", "fz", "fy", "fx", "t"]
#
# The flow field stores DISPLACEMENTS (offsets) from aligned space to raw space:
#   raw_x = aligned_x + flow_fields[0, gz, gy, gx, t]
#   raw_y = aligned_y + flow_fields[1, gz, gy, gx, t]
#   raw_z = aligned_z + flow_fields[2, gz, gy, gx, t]
#
# Grid strides (aligned space -> flow field grid):
#   - stride_x = 2048 / 128 = 16
#   - stride_y = 1328 / 83  = 16
#   - stride_z = 72 / 36    = 2
# ============================================================================

# Volume dimensions
SIZE_X, SIZE_Y, SIZE_Z, SIZE_T = 2048, 1328, 72, 7879

# Flow field grid strides (aligned space pixels per grid point)
STRIDE_X = 16
STRIDE_Y = 16
STRIDE_Z = 2
ds_flow = ts.open({
    'open': True,
    'driver': 'zarr3',
    'kvstore': f'{gs_uri}/flow_fields'
}).result()

# this is the raw data from the microscope
ds_raw = ts.open({
    'open': True,
    'driver': 'zarr3',
    'kvstore': 'gs://zapbench-release/volumes/20240930/raw/'
}).result()

In [35]:
gx = xi // STRIDE_X
gy = yi // STRIDE_Y
gz = zi // STRIDE_Z
T = 5500
offset = ds_flow[:, gz, gy, gx, T].read().result()
ioffset = np.round(offset / np.array([1, 1, 4])[:, np.newaxis]).astype(int)

In [ ]:
raw_coords = np.stack([xi, yi, zi], axis=0) + ioffset

raw_stack = ds_raw[:, :, :, T].read().result()
raw_vals = raw_stack[raw_coords[0], raw_coords[1], raw_coords[2]]

## explore a window

In [44]:
# Load segmentation
ds = ts.open({
    'open': True,
    'driver': 'zarr3',
    'kvstore': f'{gs_uri}/segmentation'
}).result()
segmentation = ds.read().result()

In [45]:
# process segmentation to build a map between voxels and cell id's

xi, yi, zi = np.where(segmentation > 0)


# get to zero based indexing
cell_id_flat = segmentation[xi, yi, zi].astype(np.uint64)
cell_id_flat -= 1


In [39]:


# this is the aligned data
ds_aligned = ts.open({
    'open': True,
    'driver': 'zarr3',
    'kvstore': 'gs://zapbench-release/volumes/20240930/aligned/'
}).result()


In [222]:
# Define a window based on a cell_id
cell_id = np.random.randint(71721)
# cell_id=52779
# cell_id=39487
cell_id=68345
print(f"{cell_id=}")

plt.figure(figsize=(20, 5))
plt.plot(act_zap[:, cell_id],
         label="zapbench"
         )
plt.plot(act_new[cell_id, :], #/ act_new[cell_id, :].mean()
         label="new"
         )
for i in CONDITION_OFFSETS:
  plt.axvline(i, color="k", ls="dotted")
plt.legend()
plt.title(f"{cell_id=}")
plt.xlim(0, 7879)
plt.xlabel("time")
plt.ylim(0, 2)
plt.ylabel("dF/F activity")

# cell_id=52779
# cell_id = 52612
# T = 5500


# cell_id=39487
# T = 3950

# T = np.argmax(act_zap[:, cell_id])

T = np.argmax(np.abs(act_zap[:, cell_id] - act_new[cell_id, :]))
w = 10
mask = cell_id_flat == cell_id
imin = xi[mask].min() - w
jmin = yi[mask].min() - w
imax = xi[mask].max() + w
jmax = yi[mask].max() + w
zmin = zi[mask].min() - 2
zmax = zi[mask].max() + 3

# Or define a random window

# window = 50
# (imin, jmin, T)=(949, 1033, 3629)
# imin = np.random.randint(0, 2048-window)
# jmin = np.random.randint(0, 1328-window)
# T = np.random.randint(7800)
# imax = imin + window
# T=np.random.randint(7879)

# jmax = jmin + window

In [223]:


print(f"{(imin, jmin, T)=}")


vol = ds_aligned[imin:imax, jmin:jmax, zmin:zmax, T].read().result()

# Get segmentation for this window, only for the z-planes we're plotting
seg_vol = segmentation[imin:imax, jmin:jmax, zmin:zmax]
seg_id = cell_id + 1  # convert to 1-indexed for segmentation lookup
cell_mask = (seg_vol == seg_id)


gx = np.arange(imin, imax) // STRIDE_X
gy = np.arange(jmin, jmax) // STRIDE_Y

raw_slices = []
rz_range = []
for z in range(zmin, zmax):
  gz = zmin // STRIDE_Z

  offset = ds_flow[
    :,
    gz,
    jmin//STRIDE_Y:jmax//STRIDE_Y,
    imin//STRIDE_X:imax//STRIDE_X, T].read().result().mean((1, 2))
  ioffset = np.round(offset / [1, 1, 4]).astype(int)

  rimin, rimax = np.array([imin, imax]) + ioffset[0]
  rjmin, rjmax = np.array([jmin, jmax]) + ioffset[1]
  rz = z + ioffset[2]
  rz_range.append(rz)
  raw_slice = ds_raw[rimin:rimax, rjmin:rjmax, rz, T].read().result()
  raw_slices.append(raw_slice)
  print(ioffset)

raw_vol = np.stack(raw_slices, axis=2)


In [224]:
# Show z-planes with cell mask

z_range = list(range(zmin, zmax))
n_cols = len(z_range)
n_rows = 4  # aligned + raw + cell mask + all segmentation

fig, axes = plt.subplots(n_rows, n_cols, figsize=(3 * n_cols, 3 * n_rows), squeeze=False)

vmin, vmax = np.percentile(vol, (1, 99))

row_labels = ["Aligned", "Raw", f"Cell {cell_id}", "All cells"]

# Row 0: aligned volume
for i, z in enumerate(z_range):
    axes[0, i].imshow(vol[:, :, i], vmin=vmin, vmax=vmax, cmap='gray')
    axes[0, i].set_title(f"z={z}")
    axes[0, i].set_xticks([])
    axes[0, i].set_yticks([])

# Row 1: raw volume
for i, z in enumerate(z_range):
    axes[1, i].imshow(raw_vol[:, :, i], vmin=vmin, vmax=vmax, cmap='gray')
    axes[1, i].set_xticks([])
    axes[1, i].set_yticks([])

# Row 2: binary mask for selected cell
for i, z in enumerate(z_range):
    axes[2, i].imshow(cell_mask[:, :, i], cmap='gray', vmin=0, vmax=1)
    axes[2, i].set_xticks([])
    axes[2, i].set_yticks([])

# Row 3: all segmented pixels with different colors
# Create consistent color mapping: map each cell ID to a color index
unique_ids = np.unique(seg_vol[seg_vol > 0])
id_to_color = {cid: (i % 20) for i, cid in enumerate(unique_ids)}

for i, z in enumerate(z_range):
    seg_slice = seg_vol[:, :, i]
    colored = np.zeros_like(seg_slice, dtype=float)
    for cid in unique_ids:
        colored[seg_slice == cid] = id_to_color[cid]
    masked_seg = np.ma.masked_where(seg_slice == 0, colored)
    axes[3, i].imshow(vol[:, :, i], vmin=vmin, vmax=vmax, cmap='gray')
    axes[3, i].imshow(masked_seg, cmap='tab20', vmin=0, vmax=19, alpha=0.7)
    axes[3, i].set_xticks([])
    axes[3, i].set_yticks([])

# Add row labels on the left
for row_idx, label in enumerate(row_labels):
    axes[row_idx, 0].set_ylabel(label, fontsize=12, rotation=0, ha='right', va='center')

plt.suptitle(f"cell_id={cell_id}, T={T}")
plt.tight_layout()